In [ ]:

from typing import List, Tuple, Optional
import cohere
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from core.config import settings
from langchain_core.prompts import ChatPromptTemplate
from tools.vectordatabase import pineretriever
from langchain_groq import ChatGroq

This should be placed inside reranker.py 

In [ ]:

co = cohere.AsyncClientV2(api_key=settings.COHERE_API_KEY)


# DOCUMENT RERANKER (unchanged logic, kept separate from the tool itself)
class DocumentReranker:
    """Wraps Cohere reranking logic."""

    def __init__(self, cohere_client, model: str = "rerank-v4.0-fast"):
        self.co = cohere_client
        self.model = model

    async def rerank(self, docs: list, query: str, top_n: int) -> List[Tuple[str, float]]:
        doc_texts = [doc.page_content.strip() for doc in docs]

        rerank_results = await self.co.rerank(
            query=query,
            documents=doc_texts,
            top_n=min(top_n, len(doc_texts)),
            model=self.model,
        )

        seen = set()
        output = []
        results_list = getattr(rerank_results, "results", rerank_results)
        for res in results_list:
            text = doc_texts[res.index]
            if text not in seen:
                seen.add(text)
                output.append((text, res.relevance_score))

        return output


reranker = DocumentReranker(cohere_client=co)



Tool to search vector db

In [ ]:
from langchain.tools import ToolRuntime

In [ ]:
import re


def strip_enrichment_header(text: str) -> str:
    """
    Removes the prepended '[Context: ...]' enrichment header from a chunk's
    text, added during ingestion to aid retrieval/reranking. The header has
    already done its job by this point (embedding + reranking), so it's
    stripped before the raw chunk is passed into the LLM's context window
    to save tokens without losing any actual medical content.
    """
    return re.sub(r"^\[Context:.*?\]\n?", "", text, flags=re.DOTALL).strip()

This is should be used inside serp_tool.py

In [ ]:
from typing import Optional, List
from langchain_core.tools import tool
from pydantic import BaseModel, Field

from core.config import settings
from tools.websearchtool import SerpRetriever

serp_api_key = settings.SERP_API_KEY


class SerpSearchInput(BaseModel):
    query: str = Field(description="Standalone, self-contained search query about the brain tumor topic.")


@tool("search_web", args_schema=SerpSearchInput)
async def search_web(
    query: str,
    runtime: ToolRuntime,
) -> str:
    """Search the web for hospitals, doctors, treatment costs, or current/local information related to brain tumor care."""
    # Safety check for missing config object
    config_dict = runtime.config or {}
    configurable = config_dict.get("configurable", {})
    
    # Extract keys safely from the configurable dictionary
    city = configurable.get("city") or config_dict.get("city")
    state = configurable.get("state") or config_dict.get("state")
    tumor_type = configurable.get("tumor_type") or config_dict.get("tumor_type")
    country = configurable.get("country") or config_dict.get("country")
    
    retriever = SerpRetriever(
        api_key=serp_api_key,
        tumor_type=tumor_type,
        city=city,
        state=state,
        country=country,
    )

    docs: List = await retriever.ainvoke(query)

    if not docs:
        return "No relevant results found on the web."


    reranked = await reranker.rerank(docs, query, top_n=3)

    if not reranked:
        return "No relevant results found after reranking."

    context_blocks = [
        f"[Context {i}] (Score: {score:.4f})\n{text}"
        for i, (text, score) in enumerate(reranked, 1)
    ]

    return "\n\n".join(context_blocks)


This will be placed inside graph.py

In [ ]:
from typing import TypedDict, Annotated, Optional, List
from langgraph.graph.message import add_messages
from langchain_core.messages import BaseMessage
from langmem.short_term import RunningSummary

class NeuroAssistState(TypedDict):
    tumor_type: str
    city: Optional[str]
    state: Optional[str]
    country: Optional[str]
    chat_summary: RunningSummary|None
    rewritten_query: Optional[str]
    cleaned_user_query:Optional[str]
    chat_agent_messages: Annotated[List[BaseMessage], add_messages]
    research_agent_messages: Annotated[List[BaseMessage], add_messages]

The agent is having trouble deciding which tool to use, so using Chain-of-Thought (CoT) might help.

In [ ]:
research_agent_system_prompt = """
<role>
Medical RAG research agent. Gather context via tools, then output a synthesized summary of findings . 
</role>
<task>Analyze the query. Call the right tool(s) with optimized, standalone search queries.</task>
<rules>
1. Location (city/state/country) is auto-injected into search_web — never ask about it or mention it's unknown.
2.STRICT GROUNDING: Rely ONLY on the clear facts directly mentioned in the retrieved context. Do not extrapolate, assume, or bring in outside medical knowledge. 
3. SUFFICIENCY OVER PERFECTION: Stop searching immediately if the gathered context contains enough information to answer the core question.
4. IMPORTANT:search_vector_db limit reached → fall back to search_web.
5. Filter out irrelevant info before summarizing.
6. Keep search queries simple and concise rather than high-level or descriptive.
</rules>

<examples>
Q: What are the causes of this tumor? Type: glioma
-> search_vector_db(query="causes of glioma")

Q: Hospitals near me for this? Type: glioma
-> search_web(query="neuro oncology brain tumor surgery hospitals")
</examples>

<Example>
<user_query>
What are the surgery and radiation options for a meningioma and what is the current treatment cost?
</user_query>

<thought_process>
- **Component 1 (Clinical):** "surgery and radiation options for a meningioma" requires established medical facts. I will query the internal vector database first. However, if the vector database tool limit is reached or returns insufficient coverage, I will fallback to web search.
- **Component 2 (Logistical/Real-world):** "current treatment cost" requires real-world pricing data, which necessitates a web search.
- **Tool Selection:** I will query internal vector database for clinical options (with web fallback planned if limits hit) and web search for cost concurrently.
</thought_process>
</Example>
"""

if the input is too big model might end up finishing its context window limit.

In [ ]:
research_agent_system_prompt = """
<role>Medical RAG agent. Gather context via tools, then output a synthesized summary.</role>
<task>Analyze query and call tools with concise, standalone search queries.</task>
<rules>
1. **Location & Context Policy:** Location is auto-injected into search_web. Never ask the user for a location. If search results contain region- or city-specific data (e.g., local hospitals, doctors, or regional pricing), adopt that location context immediately and present the data as-is without questioning or disclaiming the geographic source.
2. STRICT GROUNDING: Rely ONLY on retrieved facts. No extrapolation.
3. SUFFICIENCY: Stop searching once core info is gathered.
4. Fallback: vector_db limit/empty → use search_web.
5. Filter irrelevant info before summarizing.
Keep your internal reasoning brief, direct, and compact. Do not write out multi-step checklists, sandbox simulations, or draft summaries inside your thoughts. Go straight from tool analysis to output.
6. Keep queries simple and concise.
7. **Tagging:** In your final response, strictly format your output into these two blocks:
VECTOR DB RESULTS:
[Facts]
WEB SEARCH RESULTS:
[Facts]
</rules>
<user_query>
What are the surgery and radiation options for a meningioma and what is the current treatment cost?
</user_query>
<thought_process>
# - **Component 1 (Clinical):** "surgery and radiation options for a meningioma" requires established medical facts. I will query the internal vector database first. However, if the vector database tool limit is reached or returns insufficient coverage, I will fallback to web search.
# - **Component 2 (Logistical/Real-world):** "current treatment cost" requires real-world pricing data, which necessitates a web search.
# - **Tool Selection:** I will query internal vector database for clinical options (with web fallback planned if limits hit) and web search for cost concurrently.
# </thought_process>
</Example>
"""

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

query_rewrite_agent_system_prompt = """
You are helping answer questions about brain tumors, specifically '{tumor_type}'.
Given a chat history and the latest user question which might reference context
in the chat history, formulate a standalone question which can be understood
without the chat history and used by the research agent. Do NOT answer the question.
"""

query_rewrite_agent_prompt = ChatPromptTemplate.from_messages([
    ("system", query_rewrite_agent_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

chat_agent_prompt = ChatPromptTemplate.from_messages([
    ("system", """
<Role>You are a professional medical assistant specializing in brain tumor care and treatment guidance.</Role>

<Task>Provide clear, compassionate, and medically accurate information to patients seeking guidance about brain tumors, treatments, symptoms, and healthcare providers.</Task>

<Guidelines>
- Never use phrases like "based on the context" or "according to the provided information"
- When information is unavailable, politely acknowledge this: "I don't have specific information about that at the moment. I'd recommend consulting with your healthcare provider for personalized guidance."
- Always remind patients that your guidance is informational and should complement, not replace, professional medical consultation
- Strictly use the information provided in the research context to answer the patient's question. Do not add or create any new information.
</Guidelines>

<Research_Context>{context}</Research_Context>
"""),
    MessagesPlaceholder("chat_history"),
    ("human", "{question}"),
])

In [ ]:

from langchain_core.prompts import ChatPromptTemplate

message_summarising_agent_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
<Role>
You are a summarization agent for a medical assistant conversation.
</Role>

<Task>
Summarize the conversation while preserving medically relevant context.
The summary should retain symptoms, diagnoses, tumor type, medications,
treatment discussions, hospital/location mentions, and any ongoing concerns.
Discard greetings, acknowledgements, and repetitive conversational text.
</Task>

<Guidelines>
- Write in third person.
- Preserve all medical facts exactly.
- Do not invent information.
- Keep the summary concise.
- This summary will be used as memory for future conversations.
</Guidelines>

Previous Summary:
{summary}

New Messages:
{messages}
"""
    )
])


In [ ]:
from langchain_core.prompts import ChatPromptTemplate

tool_output_compresser_agent_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
<Role>
Your sole purpose is to prune, compress, and extract hyper-relevant information from raw tool outputs, preparing it efficiently for a downstream reasoning LLM.
</Role>

<Task>
Analyze the provided <tool_output> strictly through the lens of the <query>. Identify and extract ONLY the precise medical facts, procedure steps,  required to answer the query.
</Task>

<Guidelines>
1. **Factual Integrity**: Never alter, generalize, or summarize concrete data points. Preserve exact medical terms..
2. **Strict Extraction**: Do not synthesize a final answer for the user. Instead, extract the key data blocks verbatim or in a highly compressed list format.
3. **No Added Knowledge**: Rely strictly on the text provided inside <tool_output>. Do not add outside assumptions, external medical facts, or commentary.
4. **Extreme Brevity**: If a sentence  does not directly add value to solving the <query>, drop it entirely. 
</Guidelines>

<query>
{query}
</query>

<tool_output>
{tool_output}
</tool_output>
"""
    )
])


In [ ]:

tool_output_compresser_agent=ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.05,
)

llm=ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0.2,
)


In [ ]:

text1="""[Context 1] (Score: 0.8832)\nThis surgery makes it easier to reach and remove large macroadenomas or pituitary tumors that have spread to nearby nerves or brain tissue.\nIt also makes it easier for the surgeon to see the extent of the tumor, as well as the parts of the brain around it.\nDuring transcranial surgery, the surgeon removes the tumor through the upper part of the skull through a cut in the scalp.\n\n[Context 2] (Score: 0.8479)\nBleeding.\nBrain injury.\nDamage to the pituitary gland.\nDouble vision or loss of vision.\nEndoscopic transnasal transsphenoidal surgery and transcranial surgery are generally safe procedures.\nComplications are uncommon.\nBut as with any surgery, there are risks.\nComplications after pituitary tumor surgery can include:\nInfection.\nReaction to the medicine that puts you in a sleep-like state during surgery This sleep-like state is called anesthesia.\nTemporary headache and nasal congestion.\nThis surgery also is called a craniotomy.\nIt's used less often than endoscopic transnasal transsphenoidal surgery for pituitary tumors.\n\n[Context 3] (Score: 0.8090)\nDuring the surgery, a surgeon â€” typically a neurosurgeon partnering with a nose and sinus surgeon â€” removes the adenoma through the nose and sinuses.\nThe surgery doesn't require an external cut, also called an incision.\nIt does not affect other parts of the brain.\nThe surgery doesn't cause a scar that you can see.\nIn transnasal transsphenoidal endoscopic surgery, a surgical instrument is placed through the nostril and alongside the nasal septum to access a pituitary tumor.\nLarge macroadenomas may be hard to remove with this surgery.\nThat's particularly true if a macroadenoma has spread to nearby nerves, blood vessels or other parts of the brain.\nThis surgery also is called adenomectomy.\n\n[Context 4] (Score: 0.7618)\nCauses other symptoms, such as headache or facial pain.\nCauses the body to make too much of some hormones.\nLowers hormone levels in the body due to pressure on the pituitary gland.\nPresses on the optic nerves and limits eyesight.\nResults after surgery typically depend on the adenoma type, its size and location, and whether the tumor has grown into tissues around it.\nSurgeries to remove a pituitary tumor include endoscopic transnasal transsphenoidal surgery and craniotomy.\nSurgery to treat a pituitary tumor involves removing the tumor.\nThis is sometimes called a tumor resection.\nA surgeon may suggest surgery if a pituitary adenoma:"""
text2="""[Context 1] (Score: 0.9280)\nProton beam therapy.Another radiation option, proton beam therapy uses positively charged ions, called protons, to target tumors.\nProton beams stop after releasing their energy within the tumor.\nThis means the beams can be controlled to target a pituitary adenoma with less risk of side effects in healthy tissue.\nThis type of radiation therapy requires special equipment.\nIt isn't widely available.\nRadiation therapy can be helpful if a pituitary tumor:\nRadiation therapy uses high-energy sources of radiation to treat pituitary tumors.\nRadiation therapy can be used after surgery.\nOr it can be used alone if surgery isn't an option.\nSlightly increased risk of developing a brain tumor.\n\n[Context 2] (Score: 0.9270)\nExternal beam radiation.This method also is called fractionated radiation therapy.\nIt delivers radiation in small amounts over time.\nA series of treatments usually is done five times a week for 4 to 6 weeks.\nIntensity modulated radiation therapy.This type of radiation therapy, also called IMRT, uses a computer that allows the beams to be shaped to surround the tumor from many angles.\nThe strength of the beams can be limited.\nThat lowers the risk of side effects on healthy tissue.\nIsn't completely removed with surgery.\nMethods of radiation therapy that can be used to treat pituitary tumors include:\nPotential side effects and complications of radiation therapy for pituitary adenomas can include:\n\n[Context 3] (Score: 0.8956)\nStereotactic radiosurgery.Often delivered as a single high dose, this type of radiation therapy precisely focuses radiation beams on the tumor.\nAlthough the word \"surgery\" is in its name, no cut into the skin is needed.\nIt delivers radiation beams the size and shape of the tumor into the tumor with the aid of brain-imaging techniques.\nThis requires attaching a head frame to the skull.\nThe frame is removed right after treatment.\nLittle radiation comes in contact with healthy tissue near the tumor.\nThat lowers the risk of damage to the healthy tissue.\nThe goal of radiation therapy for pituitary adenomas is to control adenoma growth or to stop the adenoma from making hormones.\nVision changes due to damage to the optic nerves.\n\n[Context 4] (Score: 0.8526)\nMost adenomas stay in the pituitary gland or in the tissue around it, and they grow slowly.\nThey typically don't spread to other parts of the body.\nPituitary tumors can be treated in several ways.\nThe tumor may be removed with surgery.\nOr its growth may be controlled with medications or radiation therapy.\nSometimes, hormone levels are managed with medicine.\nYour health care provider may suggest a combination of these treatments.\nIn some cases, observation â€” also called a ''wait-and-see'' approach â€” may be the right choice."""
text3="""[Context 1] (Score: 0.9662)\nDuring this treatment, sticky pads are attached to the scalp.\nYou might need to shave your head so the pads can stick.\nWires connect the pads to a portable device.\nThe device generates an electrical field that hurts the glioma cells.\nSide effects of tumor treating fields therapy include skin irritation where the pads are applied to the scalp.\nTumor treating fields therapy is a treatment that uses electrical energy to hurt the glioma cells.\nThe treatment makes it hard for the cells to make new glioma cells.\nTumor treating fields therapy is used to treat an aggressive type of glioma called glioblastoma.\nThis treatment is often done at the same time as chemotherapy.\n\n[Context 2] (Score: 0.0257)\nGlioma and glioma treatment can hurt parts of the brain that help you move your body and control your thinking.\nAfter treatment you might need help to regain your ability to move, speak, see and think clearly.\nTreatments that might help include:\nOccupational therapy,which can help you get back to your daily activities, including work, after a brain tumor or other illness.\nPhysical therapy after glioma treatment can help you regain lost motor skills or muscle strength.\nPhysical therapy,which can help you regain lost motor skills or muscle strength.\nSpeech therapy,which can help if you have difficulty speaking.\nTutoring for school-age children,which can help kids cope with changes in memory and thinking after a brain tumor.\n\n[Context 3] (Score: 0.0044)\nDuring radiation therapy, you lie on a table while a machine aims energy beams at specific points on your head.\nThe beams are carefully programmed to deliver precise amounts of radiation to the glioma.\nA common schedule for radiation therapy is having treatments five days a week for a few weeks.\nFor glioma treatment, radiation therapy is often used after surgery.\nThe radiation kills any glioma cells that might remain after surgery.\nRadiation is often combined with chemotherapy.\nRadiation therapy might be the first glioma treatment if surgery isn't an option.\nRadiation uses beams of powerful energy to kill tumor cells.\nThe energy can come from X-rays, protons or other sources.\n\n[Context 4] (Score: 0.0044)\nGlioma treatment usually begins with surgery.\nBut surgery isn't always an option.\nFor example, if the glioma grows into important parts of the brain, it might be too risky to remove all of the glioma.\nOther treatments, such as radiation therapy and chemotherapy, might be recommended as the first treatment.\nWhich treatments are best for you will depend on your particular situation.\nYour health care team considers the type of glioma, its size and where it's located in the brain.\nYour treatment plan also depends on your health and your preferences.\n\n[Context 5] (Score: 0.0029)\nChemotherapy is usually used in combination with radiation therapy to treat gliomas.\nChemotherapy uses drugs to kill tumor cells.\nChemotherapy medicines are most often taken in pill form or injected into a vein.\nIn certain situations, the chemotherapy can be applied directly to the glioma cells.\nSide effects of chemotherapy depend on the type and dose of medicines you receive.\nCommon side effects include nausea and vomiting, hair loss, fever and feeling very tired.\nSome side effects may be managed with medication."""
text4="""[Context 1] (Score: 0.9988)\nIf no visible tumor remains,then no further treatment may be needed.\nBut you will have follow-up scans from time to time.\nIf the meningioma causes symptoms or shows signs that it's growing, your healthcare professional may suggest surgery.\nIf the tumor is benign and only a small piece remains,then your healthcare professional may suggest follow-up scans only.\nSome small leftover tumors may be treated with a form of radiation treatment called stereotactic radiosurgery.\nIf the tumor is irregular or cancer,you'll likely need radiation.\nSurgeons work to remove the entire meningioma.\nBut because a meningioma may be near fragile structures in the brain or spinal cord, it isn't always possible to remove the entire tumor.\n\n[Context 2] (Score: 0.9986)\nAdvances in radiation therapy increase the dose of radiation to the meningioma while giving less radiation to healthy tissue.\nRadiation therapy types for meningiomas include:\nFractionated stereotactic radiotherapy (SRT).This type gives radiation in small fractions over time, such as one treatment a day for 30 days.\nThis approach may be used for tumors too large for radiosurgery or those in an area where radiosurgery is too strong, such as near the optic nerve.\nIf the entire meningioma can't be removed surgically, your healthcare professional may suggest radiation therapy after or instead of surgery.\nIntensity-modulated radiation therapy (IMRT).This uses computer software to lower the intensity of radiation to the meningioma site.\n\n[Context 3] (Score: 0.9981)\nThis may be used for meningiomas that are near sensitive brain structures or those with a complex shape.\nProton beam radiation.This uses radioactive protons aimed right at the tumor.\nThis type lessens damage to the tissue around the tumor.\nStereotactic radiosurgery (SRS).This type of radiation treatment aims several beams of powerful radiation at a precise point.\nDespite its name, radiosurgery doesn't involve scalpels or cuts.\nRadiosurgery most often is done in an outpatient setting in a few hours.\nRadiosurgery may be a choice for people with meningiomas that can't be removed with conventional surgery or for meningiomas that come back despite treatment.\n\n[Context 4] (Score: 0.9940)\nComplications\nA meningioma and its treatment can cause long-term complications.\nTreatment most often involves surgery and radiation therapy.\nComplications may include:\nTrouble focusing.\nMemory loss.\nPersonality changes.\nSeizures.\nWeakness.\nChanges in the senses.\nTrouble with language.\nYour healthcare professional can treat some complications and refer you to specialists to help you cope with other complications.\n\n[Context 5] (Score: 0.9658)\nMedicine therapy, also called chemotherapy, rarely is used to treat meningiomas.\nBut it may be used when the meningioma doesn't respond to surgery and radiation.\nThere isn't a widely used chemotherapy approach to the treatment of meningiomas.\nBut researchers are studying other targeted approaches."""
text5="""[Context 1] (Score: 0.9993)\n[Context: Meningioma treatment: follow-up scans, surgery, stereotactic radiosurgery, or radiation based on residual tumor and growth.]\nIf no visible tumor remains,then no further treatment may be needed.\nBut you will have follow-up scans from time to time.\nIf the meningioma causes symptoms or shows signs that it's growing, your healthcare professional may suggest surgery.\nIf the tumor is benign and only a small piece remains,then your healthcare professional may suggest follow-up scans only.\nSome small leftover tumors may be treated with a form of radiation treatment called stereotactic radiosurgery.\nIf the tumor is irregular or cancer,you'll likely need radiation.\nSurgeons work to remove the entire meningioma.\nBut because a meningioma may be near fragile structures in the brain or spinal cord, it isn't always possible to remove the entire tumor.\n\n[Context 2] (Score: 0.9982)\n[Context: Alternative medicine for meningioma: acupuncture, hypnosis, massage, meditation, music therapy, relaxation exercises relieve side effects and stress, not tumor.]\nAcupuncture.\nAlternative medicine therapies that may be helpful include:\nAlternative medicine treatments don't treat meningioma.\nBut some may help give relief from treatment side effects.\nOr they might help you cope with the stress of having a meningioma.\nDiscuss choices with your healthcare professional.\nHypnosis.\nMassage.\nMeditation.\nMusic therapy.\nRelaxation exercises.\n\n[Context 3] (Score: 0.9982)\n[Context: This covers Treatment where radiation therapy advances target meningiomas, using fractionated stereotactic radiotherapy and intensity-modulated radiation therapy to increase tumor dose and spare healthy tissue.]\nAdvances in radiation therapy increase the dose of radiation to the meningioma while giving less radiation to healthy tissue.\nRadiation therapy types for meningiomas include:\nFractionated stereotactic radiotherapy (SRT).This type gives radiation in small fractions over time, such as one treatment a day for 30 days.\nThis approach may be used for tumors too large for radiosurgery or those in an area where radiosurgery is too strong, such as near the optic nerve.\nIf the entire meningioma can't be removed surgically, your healthcare professional may suggest radiation therapy after or instead of surgery.\nIntensity-modulated radiation therapy (IMRT).This uses computer software to lower the intensity of radiation to the meningioma site.\n\n[Context 4] (Score: 0.9940)\n[Context: This covers Treatment where proton beam radiation and stereotactic radiosurgery treat meningiomas near sensitive structures or complex shapes.]\nThis may be used for meningiomas that are near sensitive brain structures or those with a complex shape.\nProton beam radiation.This uses radioactive protons aimed right at the tumor.\nThis type lessens damage to the tissue around the tumor.\nStereotactic radiosurgery (SRS).This type of radiation treatment aims several beams of powerful radiation at a precise point.\nDespite its name, radiosurgery doesn't involve scalpels or cuts.\nRadiosurgery most often is done in an outpatient setting in a few hours.\nRadiosurgery may be a choice for people with meningiomas that can't be removed with conventional surgery or for meningiomas that come back despite treatment.\n\n[Context 5] (Score: 0.9927)\n[Context: This covers Treatment where monitoring and intervention decisions for meningioma are discussed]\nIf the plan is for you not to have treatment for a meningioma, you'll likely have brain scans at times to assess your meningioma and look for signs that it's growing.\nIf your healthcare provider finds that the meningioma is growing and needs to be treated, you have several treatment choices.\nNot everyone with a meningioma needs treatment right away.\nA small, slow-growing meningioma that isn't causing symptoms may not need treatment."""
text6=""""[Context 1] (Score: 0.9986)\n[Context: Chemotherapy combined with radiation for glioma: drug-induced tumor cell death, oral/IV or direct, with nausea, hair loss]\nChemotherapy is usually used in combination with radiation therapy to treat gliomas.\nChemotherapy uses drugs to kill tumor cells.\nChemotherapy medicines are most often taken in pill form or injected into a vein.\nIn certain situations, the chemotherapy can be applied directly to the glioma cells.\nSide effects of chemotherapy depend on the type and dose of medicines you receive.\nCommon side effects include nausea and vomiting, hair loss, fever and feeling very tired.\nSome side effects may be managed with medication.\n\n[Context 2] (Score: 0.9835)\n[Context: This covers Treatment where glioma management involves surgery, radiation, or chemotherapy based on tumor characteristics and patient factors.]\nGlioma treatment usually begins with surgery.\nBut surgery isn't always an option.\nFor example, if the glioma grows into important parts of the brain, it might be too risky to remove all of the glioma.\nOther treatments, such as radiation therapy and chemotherapy, might be recommended as the first treatment.\nWhich treatments are best for you will depend on your particular situation.\nYour health care team considers the type of glioma, its size and where it's located in the brain.\nYour treatment plan also depends on your health and your preferences.\n\n[Context 3] (Score: 0.9791)\n[Context: This covers Overview where glioma type informs seriousness and guides treatment options like surgery, radiation, and chemotherapy.]\nOthers happen mostly in kids.\nThe type of glioma you have helps your health care team understand how serious your condition is and what treatments might work best.\nIn general, glioma treatment options include surgery, radiation therapy, chemotherapy and others.\n\n[Context 4] (Score: 0.7909)\n[Context: This covers Treatment where radiation therapy targets glioma cells with precise energy beams, often post-surgery or combined with chemo.]\nDuring radiation therapy, you lie on a table while a machine aims energy beams at specific points on your head.\nThe beams are carefully programmed to deliver precise amounts of radiation to the glioma.\nA common schedule for radiation therapy is having treatments five days a week for a few weeks.\nFor glioma treatment, radiation therapy is often used after surgery.\nThe radiation kills any glioma cells that might remain after surgery.\nRadiation is often combined with chemotherapy.\nRadiation therapy might be the first glioma treatment if surgery isn't an option.\nRadiation uses beams of powerful energy to kill tumor cells.\nThe energy can come from X-rays, protons or other sources.\n\n[Context 5] (Score: 0.7737)\n[Context: Complementary treatments for glioma to aid coping and support alongside conventional therapy]\nAcupuncture.\nAsk your health care team if you're interested in trying complementary treatments such as:\nHypnosis.\nLittle research has been done on complementary and alternative glioma treatments.\nNo alternative treatments have been proved to cure gliomas.\nHowever, complementary treatments may help you cope with your glioma and its treatment.\nComplementary treatments also are called integrative treatments.\nThey can be used at the same time as traditional treatments, such as surgery, radiation therapy and chemotherapy.\nMeditation.\nMusic therapy.\nRelaxation exercises."""


failure_points=[text1,text2,text3,text4,text5,text6]


print("the number of tokens for each  failure point are ")
for text in failure_points:
   
    print("-----")
    print(llm.get_num_tokens(text))


In [ ]:
print(text2)

In [ ]:
query="radiation therapy options for pituitary tumors"



In [ ]:
final_prompt=tool_output_compresser_agent_prompt.format_messages(

    query=query,
    tool_output=text2
)

In [ ]:
tool_output_compresser_agent.get_num_tokens(final_prompt[0].content)

In [ ]:
# output=tool_output_compresser.invoke(
#     final_prompt
# ) 

In [ ]:
# output.content

In [ ]:
async def compress_tool_output(tool_output: str, query: str) -> str:
    final_prompt = tool_output_compresser_agent_prompt.format_messages(
        query=query,
        tool_output=tool_output,
    )

    result = await tool_output_compresser.ainvoke(final_prompt)
    return result.content  # extract the string, not the AIMessage object


class VectorSearchInput(BaseModel):
    query: str = Field(
        description="Standalone, self-contained search query about the brain tumor topic."
    )


@tool("search_vector_db", args_schema=VectorSearchInput)
async def search_vector_db(query: str, runtime: ToolRuntime) -> str:
    """
    Search the internal medical knowledge base for established facts about
    brain tumor symptoms, causes, diagnosis, treatment, and prognosis.
    """
    config_dict = runtime.config or {}
    configurable = config_dict.get("configurable", {})

    tumor_type = configurable.get("tumor_type") or config_dict.get("tumor_type")

    if not tumor_type:
        return "Error: Missing tumor type configuration context."

    search_kwargs = {
        "k": 10,
        "filter": {"tumor_type": tumor_type.lower()},
    }

    docs = await pineretriever.ainvoke(query, search_kwargs=search_kwargs)

    if not docs:
        return "No relevant documents found in the knowledge base."

    reranked = await reranker.rerank(docs, query, top_n=4)

    if not reranked:
        return "No relevant documents found after reranking."

    filtered_docs = [
        (text, score) for text, score in reranked if score >= 0.73
    ]

    # If nothing cleared the threshold, fall back to the top unfiltered
    # results so the agent can see them and judge whether its query needs
    # refining — returning an empty string gives it nothing to work with.
    docs_to_format = filtered_docs if filtered_docs else reranked

    context_blocks = [
        f"[Context {i}] (Score: {score:.4f})\n{strip_enrichment_header(text)}"
        for i, (text, score) in enumerate(docs_to_format, 1)
    ]

    tool_output = "\n\n".join(context_blocks)

    if llm.get_num_tokens(tool_output) > 550:
        tool_output = await compress_tool_output(tool_output, query)

    return tool_output

**Agent Intialisation**

In [ ]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langchain.agents.middleware.tool_call_limit import ToolCallLimitMiddleware


In [ ]:
search_vector_db_tool_limiter = ToolCallLimitMiddleware(
    tool_name="search_vector_db",
    run_limit=4, 
    thread_limit=15,
    exit_behavior="continue"
)


In [ ]:

web_tool_limiter = ToolCallLimitMiddleware(
    tool_name="search_web",
    run_limit=4, 
    thread_limit=15,
    exit_behavior="continue"
)


Reasoning models could waste tokens on internal reasoning just by overthinking.

One important think is that whenever we are making agent with multiple tools we need to make sure that context length provided by the tool output is not to big as agents go through process that requires input read + reasoning (here agent might rewrite output just to think about  its next thing)

This is one of the most critical realizations when engineering multi-tool agents.
When a tool dumps a massive payload, the agent doesn't just read it once. In every single loop, the agent has to re-read that entire payload, write an internal monologue about it, execute the next tool, and repeat. This causes an exponential token compounding effect that kills your context window and causes mid-way cutoffs. [1, 2] 
Here are the short, punchy points explaining exactly why this happens and how to protect your agent:
## 1. The Multi-Turn Compounding Tax

* The Problem: Tool outputs are not a one-time cost.
* The Reality: If a web search returns 5,000 tokens of data on Turn 1, and your agent takes 4 more turns to solve the problem, you pay that 5,000-token tax 4 separate times as input in the execution loop.

## 2. The Internal "Echo Chamber" Loop

* The Problem: Agents are text-predictors that think out loud.
* The Reality: When forced to look at a massive tool payload, the agent will rewrite, summarize, or quote chunks of that data in its internal reasoning just to "plan" its next step. This duplicates the heavy tool data inside its own thinking space. [3, 4] 

## 3. Tool Output Aggression (JSON/HTML Bloat)

* The Problem: Raw tools are built for software, not LLMs.
* The Reality: A database query or API call returns boilerplate metadata, nested arrays, headers, and keys that the agent does not need, which clogs the attention mechanism. [5] 

## 4. Golden Rules for Tool Defensive Engineering
To fix this, you must build firewalls between your tools and your agent:

* The Extraction Layer: Never let a tool talk directly to the agent. Pass tool outputs through a Python cleaner or a tiny, cheap model (like GPT-4o-mini) to extract only the specific answer before the main agent sees it. [6] 
* Aggressive Truncation: Hard-cap your tool outputs to a maximum of 800–1,000 tokens. If a web scraper pulls a 10-page article, chop it down to the top snippets.
* Markdown Flattening: Strip away all JSON braces, keys, and HTML tags from tool outputs. Convert the data into simple, flat Markdown bullet points before injection.




In [ ]:
llm=ChatGroq(
    model="qwen/qwen3.6-27b",
    temperature=0.2,
)
research_agent=create_agent(
    model=llm,
    tools=[search_vector_db, search_web],
    system_prompt=research_agent_system_prompt,
    middleware=[search_vector_db_tool_limiter, web_tool_limiter],
    debug=True
)

In [ ]:
query_rewrite_agent=ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.08,
)

In [ ]:
chat_agent=ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.08,
    streaming=True
)

In [ ]:
chat_sumarising_agent=ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.05,
)

In [ ]:
tool_output_compresser=ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.05,
)

Here i have collected the vector db tool output where the model either was not able to analyse properly or the output stopped because of finish reason length .

In [ ]:


# --- Test the logic ---
# medical_query = "Hi, my name is John Doe (user: jdoe99). I am 72 years old and experiencing sudden chest tightness."
# safe_query = sanitize_identity_only(medical_query)

# print(safe_query)
# Output: "Hi, my name is <PERSON> (<USERNAME>). I am 72 years old and experiencing sudden chest tightness."


In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
from aigraph.state import NeuroAssistState
from utils.presidio_worker import execute_sanitize

# Create an explicit thread pool capped at 1 or 2 workers for your free tier CPU
# This prevents your application from consuming too much memory under load
safety_thread_executor = ThreadPoolExecutor(max_workers=1)

async def query_safety_check(state: NeuroAssistState):
    # Extract the last message string from state
    user_query = state["chat_agent_messages"][-1].content

    # Fetch active async event loop
    loop = asyncio.get_running_loop()
    
    # Hand off the work to your custom explicit ThreadPoolExecutor
    cleaned_user_query = await loop.run_in_executor(
        safety_thread_executor, 
        execute_sanitize, 
        user_query
    )

    # Return key-value changes to LangGraph state reducer
    return {"cleaned_user_query": cleaned_user_query}

In [ ]:
import re
from langchain_core.output_parsers import BaseOutputParser

class CleanQueryParser(BaseOutputParser[str]):
    def parse(self, text: str) -> str:
        # Erase everything inside <think>...</think> blocks safely
        cleaned = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL)
        return cleaned.strip()

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableConfig

query_rewrite_chain = query_rewrite_agent_prompt | query_rewrite_agent| CleanQueryParser() 

async def query_rewrite_node(state: NeuroAssistState, config: RunnableConfig):
    chat_history = state["chat_agent_messages"]

    if not chat_history:
        return {}

    current_question = state.get('cleaned_user_query',state["chat_agent_messages"][-1])
    prior_history = chat_history[-5:-1]

    # No prior turns — nothing to resolve against, skip reformulation
    if not prior_history:
        return {"rewritten_query": current_question}

    reformulated_question = await query_rewrite_chain.ainvoke(
        {
            "chat_history": prior_history,
            "input": current_question,
            "tumor_type": state["tumor_type"],
        },
        config=config,
    )

    return {"rewritten_query": reformulated_question}

In [ ]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig

async def research_node(state: NeuroAssistState, config: RunnableConfig):
    recent_chat_history = state["chat_agent_messages"][-3:]
    
    # Guard against empty chat history to prevent IndexError
    if not recent_chat_history:
        return {"research_agent_messages": []}

    # Extract the latest query from the last message
    last_message = recent_chat_history[-1]

    query = state.get("rewritten_query", last_message.content)


    # Construct the input messages without duplicating the last message
    input_messages = [
        HumanMessage(content=f"Query: {query}\n Brain tumor type: {state['tumor_type']}")
    ] 
    
    current_configurable = config.get("configurable", {})

    sub_agent_config: RunnableConfig = {
        **config,
        "configurable": {
            **current_configurable,
            "city": state.get("city"),
            "state": state.get("state"),
            "country": state.get("country"),
            "tumor_type": state.get("tumor_type"),
        }
    }

    result = await research_agent.ainvoke(
        {"messages": input_messages},
        config=sub_agent_config
    )

    # tool-call trace via add_messages, useful for logging/debugging
    return {"research_agent_messages": result["messages"]}

In [ ]:
from langmem.short_term import summarize_messages, RunningSummary

In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableConfig
from langmem.short_term import summarize_messages   # import may vary depending on your version


async def chat_node(state: NeuroAssistState, config: RunnableConfig):
    # Current user question is the latest message
    question = state["chat_agent_messages"][-1].content

    # Everything before that is conversation history
    chat_history = state["chat_agent_messages"][:-1]

    # ---------- Langmem SUMMARIZATION LOGIC ----------
    summarization_result =summarize_messages(
        chat_history,
        running_summary=state.get("chat_summary"),
        token_counter=chat_agent.get_num_tokens_from_messages,
        model=chat_sumarising_agent,
        max_tokens=300 ,
        max_tokens_before_summary=800,
        max_summary_tokens=128,
        final_prompt=message_summarising_agent_prompt
    )

    # These are the messages that should go into the prompt
    chat_history = summarization_result.messages
    # ---------------------------------------------

    # Pull latest research response
    research_messages = state.get("research_agent_messages", [])
    context = ""
    for msg in reversed(research_messages):
        if isinstance(msg, AIMessage) and msg.content:
            context = msg.content
            break

    prompt_messages = chat_agent_prompt.format_messages(
        context=context,
        chat_history=chat_history,
        question=question,
    )

    response = await chat_agent.ainvoke(prompt_messages, config=config)

    return {
        "chat_agent_messages": [AIMessage(content=response.content)],
        "chat_summary": summarization_result.running_summary,
    }

In [ ]:
from langgraph.graph import StateGraph, START, END


graph_builder = StateGraph(NeuroAssistState)


# Creating nodes for graph
graph_builder.add_node("research_node", research_node)
graph_builder.add_node("query_rewrite_node", query_rewrite_node)
graph_builder.add_node("query_safety_check",query_safety_check)
graph_builder.add_node("chat_node", chat_node)



# connecting nodes
graph_builder.add_edge(START, "query_safety_check")
graph_builder.add_edge("query_safety_check", "query_rewrite_node")
graph_builder.add_edge("query_rewrite_node", "research_node")
graph_builder.add_edge("research_node", "chat_node")
graph_builder.add_edge("chat_node", END)



In [ ]:
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver

from psycopg.rows import dict_row
from psycopg_pool import AsyncConnectionPool


LANGGRAPH_DB_URL=""

In [ ]:
pool = AsyncConnectionPool(
    conninfo=LANGGRAPH_DB_URL,
    min_size=1,
    max_size=5,
    open=False, # We open it manually on the next line
    check=AsyncConnectionPool.check_connection,
    kwargs={"autocommit": True, "row_factory": dict_row}
)

await pool.open()
checkpointer = AsyncPostgresSaver(pool)

In [ ]:
graph = graph_builder.compile(checkpointer=checkpointer)

In [ ]:
from IPython.display import Image, display

# Assuming 'app' or 'graph' is your compiled LangGraph workflow
display(Image(graph.get_graph().draw_mermaid_png()))


In [ ]:
from langchain_core.messages import HumanMessage
import uuid

config = {
    "configurable": {
        "user_id": "test_user_12345",
        "thread_id": str(uuid.uuid4()),
    }
}


In [ ]:
break

In [ ]:

state = {
    "tumor_type": "glioma",
    "city": "Pune",
    "state": "Maharashtra",
    "country": "India",
    "chat_agent_messages": [
        HumanMessage(content="What are the  causes for glioma and how can it be treated?")
    ],
    "research_agent_messages": [],
}



In [ ]:
async for chunk_msg, metadata in graph.astream(
    state, 
    config=config, 
    stream_mode="messages" 
):
    # 4. Filter for chunks that come specifically from your chat_node
    if metadata.get("langgraph_node") == "chat_node":
        token = chunk_msg.content
        if token:
            # Prints tokens in real-time right inside the notebook cell
            print(token, end="", flush=True)
            
print("\n\n--- Stream Finished ---")

In [ ]:


# result_1 = await graph.ainvoke(state, config=config)

# print("========== TURN 1 ==========")
# print("--- RESEARCH AGENT MESSAGES ---")
# for msg in result_1["research_agent_messages"]:
#     print(f"[{msg.type}] {msg.content}\n")



In [ ]:

state_2 = {
    
    "chat_agent_messages":[
        HumanMessage(content=" is it okay if i leave it as it is , will it cause any side effects or any kind of pain?")
    ],
   
}

result_2 = await graph.ainvoke(state_2, config=config)

print("========== TURN 2 ==========")
print("--- RESEARCH AGENT MESSAGES ---")
for msg in result_2["research_agent_messages"]:
    print(f"[{msg.type}] {msg.content}\n")

print("--- CHAT AGENT MESSAGES ---")
for msg in result_2["chat_agent_messages"]:
    print(f"[{msg.type}] {msg.content}\n")

**Evaluation**

what i observed is changing llm also helps if react agent is not working well.

**Creating a separate graph containing research agent only , to do its evaluation.**

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver


builder = StateGraph(NeuroAssistState)


# Creating nodes for graph
builder.add_node("research_node", research_node)

# connecting nodes
builder.add_edge(START, "research_node")
builder.add_edge("research_node", END)



In [ ]:
checkpointer = InMemorySaver()
test_graph = builder.compile(checkpointer=checkpointer)

**Agent Evaluation**

In [ ]:
# Fixed import location for modern LangChain
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
import uuid
import asyncio
from langsmith import Client, aevaluate
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
from agentevals.trajectory.llm import create_async_trajectory_llm_as_judge, TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE
from openevals.llm import create_async_llm_as_judge
from openevals.prompts import CORRECTNESS_PROMPT
import pickle
from langchain.agents.structured_output import ToolStrategy

In [ ]:
eval_inputs = [
    {
        "inputs": {
            "tumor_type": "pituitary",
            "city": "Mumbai",
            "country": "India",
            "state": "Maharashtra",
            "question": "What are the specialized hospitals and surgical treatment options available for pituitary tumors in Mumbai?"
        },
        "outputs": {
            "ground_truth": "Specialized surgical treatment options for pituitary tumors in Mumbai include minimally invasive endoscopic transsphenoidal surgery, performed by expert neurosurgeons at premier medical institutions such as Tata Memorial Hospital, Jaslok Hospital, and Hinduja Hospital.",
            "messages": [
                HumanMessage(content="Query: What are the specialized hospitals and surgical treatment options available for pituitary tumors in Mumbai?\n Brain tumor type: pituitary"),
                AIMessage(content="", tool_calls=[
                    {"id": "pituitary_web_01", "name": "search_web", "args": {"query": "specialized hospitals and pituitary tumor surgery options Mumbai"}}
                ]),
                ToolMessage(
                    content="Title: Pituitary Tumor Treatment and Neurosurgery in Mumbai\nDescription: Leading hospitals in Mumbai like Tata Memorial Hospital, Jaslok Hospital, and Hinduja Hospital offer advanced surgical options including endoscopic transsphenoidal surgery and specialized endocrinology care for pituitary adenomas.",
                    tool_call_id="pituitary_web_01"
                ),
                AIMessage(content="Specialized surgical treatment options for pituitary tumors in Mumbai include endoscopic transsphenoidal surgery performed by expert neurosurgical teams. Leading institutions providing these advanced procedures and comprehensive endocrinology care include Tata Memorial Hospital, Jaslok Hospital, and Hinduja Hospital.")
            ]
        }
    },
    {
        "inputs": {
            "tumor_type": "pituitary",
            "city": "Mumbai",
            "country": "India",
            "state": "Maharashtra",
            "question": "Where can I find top endocrinologists and hormone specialists for managing pituitary adenomas in Mumbai?"
        },
        "outputs": {
            "ground_truth": "Top endocrinologists and hormone specialists for managing pituitary tumors in Mumbai can be found at renowned multi-specialty and research centers such as Kokilaben Dhirubhai Ambani Hospital, Lilavati Hospital, and Jaslok Hospital.",
            "messages": [
                HumanMessage(content="Query: Where can I find top endocrinologists and hormone specialists for managing pituitary adenomas in Mumbai?\n Brain tumor type: pituitary"),
                AIMessage(content="", tool_calls=[
                    {"id": "pituitary_web_02", "name": "search_web", "args": {"query": "top endocrinologists and pituitary tumor specialists Mumbai hospitals"}}
                ]),
                ToolMessage(
                    content="Title: Endocrinology and Neuro-Endocrine Specialists in Mumbai\nDescription: Patients seeking hormone management and treatment for pituitary adenomas can consult expert endocrinologists at top-tier facilities like Kokilaben Dhirubhai Ambani Hospital, Lilavati Hospital, and Jaslok Hospital.",
                    tool_call_id="pituitary_web_02"
                ),
                AIMessage(content="Expert endocrinologists and hormone disorder specialists for managing pituitary adenomas in Mumbai practice at premier healthcare institutions including Kokilaben Dhirubhai Ambani Hospital, Lilavati Hospital, and Jaslok Hospital.")
            ]
        }
    },
    {
        "inputs": {
            "tumor_type": "pituitary",
            "city": "Mumbai",
            "country": "India",
            "state": "Maharashtra",
            "question": "What is the approximate cost of transsphenoidal surgery for a pituitary tumor in Mumbai hospitals?"
        },
        "outputs": {
            "ground_truth": "The approximate cost of transsphenoidal surgery for a pituitary tumor in Mumbai ranges between ₹2,00,000 and ₹5,50,000, varying across private and government-subsidized medical centers.",
            "messages": [
                HumanMessage(content="Query: What is the approximate cost of transsphenoidal surgery for a pituitary tumor in Mumbai hospitals?\n Brain tumor type: pituitary"),
                AIMessage(content="", tool_calls=[
                    {"id": "pituitary_web_03", "name": "search_web", "args": {"query": "pituitary tumor transsphenoidal surgery cost Mumbai hospitals"}}
                ]),
                ToolMessage(
                    content="Title: Pituitary Tumor Surgery Cost in Mumbai\nDescription: The average cost for transsphenoidal pituitary tumor resection in Mumbai ranges from INR 2,00,000 to INR 5,50,000 depending on hospital room categories, surgeon fees, and whether treatment is done at a private trust or public institute.",
                    tool_call_id="pituitary_web_03"
                ),
                AIMessage(content="The approximate cost for pituitary tumor transsphenoidal surgery in Mumbai typically ranges from ₹2,00,000 to ₹5,50,000, depending on the choice of hospital, room tier, and whether public or private facilities are utilized.")
            ]
        }
    }
]

In [ ]:
dataset_name = "websearchtool_evaluation_dataset"
tumor_type = "pituitary"

In [ ]:


# with open(f"EvaluationData/{tumor_type}/{dataset_name}.pkl", "wb") as f:
#     pickle.dump(eval_inputs, f)
 

In [ ]:
# with open(f"EvaluationData/{tumor_type}/{dataset_name}.pkl", "rb") as f:
#     loaded_dataset = pickle.load(f)

Function to run the graph

In [ ]:
async def target_graph_runner(inputs: dict):
    """
    This wrapper acts as the bridge. It receives the dataset row input, 
    invokes the graph, and returns the message list the judge needs to evaluate.
    """
    config = {
        "configurable": {
            "user_id": "test_user_12345",
            "thread_id": str(uuid.uuid4()),
        }
    }

    eval_graph_state = {
        "tumor_type": inputs.get("tumor_type", "meningioma"),
        "city": inputs.get("city", "Pune"),
        "state": inputs.get("state", "Maharashtra"),
        "country": inputs.get("country", "India"),
        "chat_agent_messages": [
            HumanMessage(content=inputs["question"])
        ],
        "research_agent_messages": [],
    }

    # Execute the graph
    result = await test_graph.ainvoke(eval_graph_state, config=config)

    # Return what the evaluator expects as the 'output'
    return {"research_agent_messages": result["research_agent_messages"], 
            "final_answer": result["research_agent_messages"][-1].content}

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class WebDataEvaluation(BaseModel):
    comment: str = Field(
        description=(
            "A structured, objective engineering summary detailing exactly which specific facts, URLs, "
            "or data points from live web search verified, logically justified, or disproved the claim. "
            "This field serves as the explicit rationale payload for downstream validation agents."
        )
    )
    score: Literal[0, 1] = Field(
        description=(
            "Binary evaluation flag encoded as an integer. "
            "Set to 1 if the claim is fully SUPPORTED or constitutes a valid clinical/institutional logical deduction "
            "based on the search data. "
            "Set to 0 only if the claim is explicitly CONTRADICTED or completely unrelated."
        )
    )

When agent is not able to make its thought process clearly COT helps.

In [ ]:
web_data_evaluator_system_prompt = """
<role>Evaluate strict factual grounding. Focus entirely on whether the specific facts stated in the answer are anchored in the search context. Do not penalize for missing topic keywords or generalized directory data.</role>

<Search Context>
{context}
</Search Context>

<Question>
{human_input}
</Question>

<Answer>
{answer}
</Answer>

<instructions>
1. Trace every factual assertion (names, ratings, locations, specialties) in the answer back to the search context.
2. Score = 1 if the entities and facts are grounded in the context (either explicitly or via standard medical/directory hierarchy).
3. Score = 0 only if specific facts/entities are fabricated or contradicted.
</instructions>

<medical_domain_rules>
- **Grounding & Hierarchy Rule:** If a hospital, institution, or specialist is verified in the context, framing them as capable of handling brain conditions/tumors is fully grounded based on their professional classification. Do not require the exact word "{tumor_type}" or "brain tumor" to appear in the text snippets.
- Treat directory metadata (addresses, names, ratings) as valid, grounded supporting evidence.
</medical_domain_rules>

<example>
<context>
Jaslok Hospital & Research Centre: Rating 4.8, Peddar Rd, Mumbai. General multi-specialty care facility.
</context>
<answer>
Jaslok Hospital on Peddar Rd, Mumbai has a 4.8 rating and handles complex neurological and brain tumor conditions.
</answer>
<thought_process>
- Entities (Jaslok Hospital, Peddar Rd, Mumbai, 4.8 rating) are directly present in the context.
- The claim of offering advanced care for brain conditions is justified by its multi-specialty research center classification under medical hierarchy rules. 
- Therefore, the facts are grounded.
</thought_process>
<score>1</score>
</example>

Return:
- score: 0 or 1
- comment: Explain your reasoning, showing how the claims are grounded via direct match or medical hierarchy from the search context.
"""

web_eval_agent_prompt_template = ChatPromptTemplate.from_messages([
    ("system", web_data_evaluator_system_prompt),
    ("human", "{human_input}"),
])

In [ ]:

async_judge = create_async_trajectory_llm_as_judge(
    model="groq:llama-3.3-70b-versatile",
    prompt=TRAJECTORY_ACCURACY_PROMPT_WITH_REFERENCE,
)

correctness_evaluator = create_async_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="correctness",
    model="groq:openai/gpt-oss-120b"
)




In [ ]:

# gpt oss model is giving error so we need to invoke it seprately for pydantic response parsing.

base_llm=ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.05,
)

structuring_llm=base_llm.with_structured_output(WebDataEvaluation)


In [ ]:
# human_input='What are the top hospitals for treating a glioma?'
# context="""[Context 1] (Score: 0.0019)\nTitle: Jaslok Hospital & Research Centre\nRating: 4.8 (12K)\nType: Hospital\nPhone: 099301 92000\nAddress: 15, Dr Gopalrao Deshmukh Marg\nHours: Open 24 hours\nDescription: \"One of the best treatments provided and the results were amazing.\"\nLatitude: 18.97166, Longitude: 72.809832\nWebsite: https://www.jaslokhospital.net/\nDirections: https://www.google.com/maps/dir//Jaslok+Hospital+%26+Research+Centre,+15,+Dr+Gopalrao+Deshmukh+Marg,+Peddar+Rd,+Mumbai,+Maharashtra+400026/data=!4m6!4m5!1m1!4e2!1m2!1m1!1s0x3be7ce76d1f47593:0x73af47ca4e2965c7?sa=X&ved=2ahUKEwivwLDs6YuWAxXMHoYAHeFLKXEQ48ADegQINRAA&hl=en&gl=in\n\n[Context 2] (Score: 0.0015)\nTitle: Sir H. N. Reliance Foundation Hospital and Research Centre\nRating: 4.8 (13K)\nType: Private hospital\nPhone: 1800 890 1111\nAddress: Raja Ram Mohan Roy Rd\nHours: Open · Closes 8 pm\nDescription: \"professional and experienced doctors and got best treatment.\"\nLatitude: 18.958899, Longitude: 72.819948\nWebsite: https://www.rfhospital.org/\nDirections: https://www.google.com/maps/dir//Sir+HN+Reliance+Foundation+Hospital+and+Research+Centre,+Raja+Ram+Mohan+Roy+Rd,+Prathna+Samaj,+Girgaon,+Mumbai,+Maharashtra+400004/data=!4m6!4m5!1m1!4e2!1m2!1m1!1s0x3be7ce11624bd7f5:0x9e69b276b442c51a?sa=X&ved=2ahUKEwivwLDs6YuWAxXMHoYAHeFLKXEQ48ADegQILhAA&hl=en&gl=in\n\n[Context 3] (Score: 0.0009)\nTitle: Gleneagles Hospital\nRating: 4.8 (23K)\nType: Hospital\nPhone: 092402 61611\nAddress: 35, Dr Ernest Borges Rd, opp. Shirodkar High School\nHours: Open 24 hours\nDescription: \"... service and doctors with state of the art technology treatment.\"\nLatitude: 18.999504, Longitude: 72.840605\nWebsite: https://www.gleneagleshospitals.co.in/gleneagles-hospital-parel-mumbai\nDirections: https://www.google.com/maps/dir//Gleneagles+Hospital,+35,+Dr+Ernest+Borges+Rd,+opp.+Shirodkar+High+School,+Parel+East,+Parel,+Mumbai,+Maharashtra+400012/data=!4m6!4m5!1m1!4e2!1m2!1m1!1s0x3be7cef0ae54abc5:0xdbda4f056a447a6d?sa=X&ved=2ahUKEwivwLDs6YuWAxXMHoYAHeFLKXEQ48ADegQIOBAA&hl=en&gl=in"""
# tumor_type="glioma"
# answer="""Based on the search results, here are some of the top-rated hospitals for treating glioma in your area:\n\n*   **Jaslok Hospital & Research Centre**\n    *   **Rating:** 4.8/5 (12,000+ reviews)\n    *   **Highlights:** Patients frequently praise the hospital for providing excellent treatment and achieving amazing results.\n    *   **Location:** 15, Dr Gopalrao Deshmukh Marg, Peddar Rd, Mumbai.\n\n*   **Sir H. N. Reliance Foundation Hospital and Research Centre**\n    *   **Rating:** 4.8/5 (13,000+ reviews)\n    *   **Highlights:** Known for having professional and experienced doctors who provide top-tier treatment.\n    *   **Location:** Raja Ram Mohan Roy Rd, Girgaon, Mumbai.\n\n*   **Gleneagles Hospital**\n    *   **Rating:** 4.8/5 (23,000+ reviews)\n    *   **Highlights:** Recognized for its state-of-the-art technology and high-quality service.\n    *   **Location:** 35, Dr Ernest Borges Rd, Parel, Mumbai.\n\nThese facilities are highly regarded for their neuro-oncology and neurosurgery departments, offering advanced care for brain tumor patients."""
# answer=structuring_llm.invoke(web_eval_agent_prompt_template.format_messages(context=context,tumor_type=tumor_type,human_input=human_input,answer=answer)).model_dump()

In [ ]:
import re


def extract_web_answer(agent_answer: str) -> str:
    """Extracts the content inside <web>...</web> tags from the agent's answer.
    Returns an empty string if the tag is not present."""
    match = re.search(r"<web>(.*?)</web>", agent_answer, flags=re.DOTALL)
    return match.group(1).strip() if match else agent_answer


def extract_vectordb_answer(agent_answer: str) -> str:
    """Extracts the content inside <vectordb>...</vectordb> tags from the agent's answer.
    Returns an empty string if the tag is not present."""
    match = re.search(r"<vectordb>(.*?)</vectordb>", agent_answer, flags=re.DOTALL)
    return match.group(1).strip() if match else agent_answer

Factual correctness evaluation

In [ ]:
import uuid
from langsmith.evaluation import EvaluationResult
from langchain_core.messages import HumanMessage


def _get_called_tools(messages) -> set[str]:
    """Scan agent trajectory messages and collect the names of tools that were actually invoked."""
    called = set()
    for msg in messages:
        tool_calls = getattr(msg, "tool_calls", None)
        if tool_calls:
            for tc in tool_calls:
                name = tc.get("name") if isinstance(tc, dict) else getattr(tc, "name", None)
                if name:
                    called.add(name)
    return called


pituitary_vectorandwebsearchtool_correctness_evaluation_dataset

In [ ]:


async def _run_web_eval(run, example, agent_answer) -> tuple:
    """
    Evaluates the agent's answer against the information actually returned
    by the search_web tool during execution.
    """

    runtime_messages = run.outputs["research_agent_messages"]

    web_context = []

    for msg in runtime_messages:
        # ToolMessage from search_web
        if getattr(msg, "name", None) == "search_web":
            web_context.append(str(msg.content))

    context = "\n\n".join(web_context)

    question = example.inputs["question"]


    web_answer_only = extract_web_answer(agent_answer)


    # extract from web tag and then send in agent .

    result = (
        await structuring_llm.ainvoke(
            web_eval_agent_prompt_template.format_messages(
                context=context,
                tumor_type=example.inputs.get("tumor_type"),
                human_input=question,
                answer=web_answer_only,
            )
        )
    ).model_dump()

    return (
        result.get("score"),
        result.get("reasoning") or result.get("comment"),
    )


In [ ]:
async def _run_vectordb_eval(run, example, agent_answer) -> tuple:

     reference_outputs = example.outputs["ground_truth"]
     input_query = example.inputs["question"]


     vectordb_answer_only = extract_vectordb_answer(agent_answer)

     eval_result = await correctness_evaluator(
                inputs=input_query,
                outputs=vectordb_answer_only,
                reference_outputs=reference_outputs,
            )

     score_value = eval_result.get("score")
     comment_value = eval_result.get("reasoning") or eval_result.get("comment")

     return eval_result


**These fuctions are explicilty written for third case evalucation where agent is supposed to call both the tools.The reason i passed ai queries is that i would just require extra gent just to classify quereis to know which query is menat for which judge agent**

In [ ]:
def get_toolcall_queries_text(messages, tool_name):
    queries = []

    for message in messages:
        tool_calls = getattr(message, "tool_calls", [])

        for tool_call in tool_calls:
            if tool_call.get("name") == tool_name:  
                query = tool_call.get("args", {}).get("query")
                if query:
                    queries.append(query)

    return "\n".join(
        f"{i}) {query}" for i, query in enumerate(queries, start=1)
    )

In [ ]:
import re

def parse_agent_response(response_text: str) -> dict:
    """
    Parses the agent's text response using RegEx and returns a dictionary 
    with the extracted sections mapped to 'vectordb' and 'web'.
    """
    # RegEx pattern to cleanly split the two sections using DOTALL (. matches newlines)
    pattern = re.compile(
        r"VECTOR DB RESULTS:\s*(.*?)\s*WEB SEARCH RESULTS:\s*(.*)",
        re.DOTALL | re.IGNORECASE
    )

    match = pattern.search(response_text)

    if match:
        vector_db_data = match.group(1).strip()
        web_search_data = match.group(2).strip()
        
        # Return as a dictionary with the requested keys, wrapped in tags for evaluation
        return {
            "vectordb": vector_db_data,
            "web": web_search_data
        }
    else:
        # Fallback or empty return if parsing fails
        raise ValueError("Parsing failed: Expected headers ('VECTOR DB RESULTS:' and 'WEB SEARCH RESULTS:') not found.")



In [ ]:


async def _run_web_eval_dual(run, example, agent_answer) -> tuple:
    """
    Evaluates the agent's answer against the information actually returned
    by the search_web tool during execution.
    """

    runtime_messages = run.outputs["research_agent_messages"]



    web_context = []

    for msg in runtime_messages:
        # ToolMessage from search_web
        if getattr(msg, "name", None) == "search_web":
            web_context.append(str(msg.content))

    context = "\n\n".join(web_context)

    question = get_toolcall_queries_text(runtime_messages,"search_web")


    web_answer_only = parse_agent_response(agent_answer)["web"]


    # extract from web tag and then send in agent .

    result = (
        await structuring_llm.ainvoke(
            web_eval_agent_prompt_template.format_messages(
                context=context,
                tumor_type=example.inputs.get("tumor_type"),
                human_input=question,
                answer=web_answer_only,
            )
        )
    ).model_dump()

    return (
        result.get("score"),
        result.get("reasoning") or result.get("comment"),
    )


In [ ]:
async def _run_vectordb_eval_dual(run, example, agent_answer) -> tuple:

     reference_outputs = example.outputs["ground_truth"]

     runtime_messages = run.outputs["research_agent_messages"]
    

     input_query = get_toolcall_queries_text(runtime_messages,"search_vector_db")


     vectordb_answer_only = parse_agent_response(agent_answer)["vectordb"]

     eval_result = await correctness_evaluator(
                inputs=input_query,
                outputs=vectordb_answer_only,
                reference_outputs=reference_outputs,
            )

     score_value = eval_result.get("score")
     comment_value = eval_result.get("reasoning") or eval_result.get("comment")

     return eval_result

In [ ]:


async def correctness_eval_bridge(run, example) -> EvaluationResult:
    agent_answer = run.outputs["final_answer"]
    reference_outputs = example.outputs["ground_truth"]
    input_query = example.inputs["question"]

    runtime_messages = run.outputs["research_agent_messages"]
    called_tools = _get_called_tools(runtime_messages)

    used_vector_db = "search_vector_db" in called_tools
    used_web_search = "search_web" in called_tools

    # Case 1: only vector DB was used -> static reference data is authoritative.
    if used_vector_db and not used_web_search:

        eval_result = await _run_vectordb_eval(run, example, agent_answer)
      
        score_value = eval_result.get("score")
        comment_value = eval_result.get("reasoning") or eval_result.get("comment")

        return EvaluationResult(
            key="correctness",
            score=score_value,
            comment=f"[Vector-DB only] {comment_value}",
        )

    # Case 2: only web search was used -> static ground_truth isn't applicable here
    # (it wasn't sourced from the reference data), so skip correctness_evaluator
    # entirely and rely on the web evaluator agent alone.
    if used_web_search and not used_vector_db:
        web_score, web_comment = await _run_web_eval(
             run,
             example,
             agent_answer,
             )

        return EvaluationResult(
            key="correctness",
            score=web_score,
            comment=f"[Web-search only] {web_comment}",
        )

    # Case 3: both vector DB and web search were used -> run static check first,
    # then verify against web trajectory too, and combine.
    if used_vector_db and used_web_search:

        eval_result = await _run_vectordb_eval_dual(run, example, agent_answer)
       
        score_value = eval_result.get("score")
        comment_value = eval_result.get("reasoning") or eval_result.get("comment")

        web_score, web_comment = await _run_web_eval_dual(run, example, agent_answer)

        # Combine: require both signals to agree the claim is correct.
        if score_value is not None and web_score is not None:
            combined_score = min(score_value, web_score)
        else:
            combined_score = score_value if web_score is None else web_score

        combined_comment = (
            f"[Reference DB] {comment_value} | [Web Verification] {web_comment}"
        )

        return EvaluationResult(
            key="correctness",
            score=combined_score,
            comment=combined_comment,
        )

    # Case 4: neither tool was called -> unverifiable, treat as hallucination risk.
    return EvaluationResult(
        key="correctness",
        score=0,
        comment=(
            "HALLUCINATION_RISK: No search_vector_db or search_web tool call found "
            "in trajectory. Claim is unverified."
        ),
    )

Trajectory Evaluation

In [ ]:

async def trajectory_eval_bridge(run, example) -> dict:
    """
    LangSmith runs this callback function to score each sample.
    """
    # Extract historical runtime output messages vs golden reference data
    runtime_messages = run.outputs["research_agent_messages"]
    golden_messages = example.outputs["messages"]  # Adjusted to point to standard outputs mapping
    
    # Execute your AgentEvals judge
    evaluation = await async_judge(
        outputs=runtime_messages, 
        reference_outputs=golden_messages
    )
    
    # Map the outcome format so the LangSmith UI understands it
    return {
        "key": "trajectory_accuracy",
        "score": int(evaluation["score"]), # Converted to int to cleanly plot 1s and 0s in UI
        "comment": evaluation.get("comment")
    }


In [ ]:
# async def main():
#     client = Client()
    
#     # 1. Ensure the dataset exists
#     data_name=f"{tumor_type}_{dataset_name}"
#     if not client.has_dataset(dataset_name=data_name):
#         dataset = client.create_dataset(dataset_name=data_name)
#     else:
#         # Fetch the existing dataset to get its ID
#         dataset = client.get_dataset(dataset_name=data_name)

#     # 2. Append new examples (Runs every time)
#     client.create_examples(
#         inputs=[e["inputs"] for e in loaded_dataset], 
#         outputs=[e["outputs"] for e in loaded_dataset], 
#         dataset_id=dataset.id
#     )

# await main()

In [ ]:
dataset_name = "vectorsearchtool_evaluation_dataset"
tumor_type = "pituitary"

In [ ]:
async def  evaluate_dataset(dataset_name: str, tumor_type: str):

      data_name=f"{tumor_type}_{dataset_name}"
      
      await aevaluate(
            target_graph_runner,
            data=data_name,
            evaluators=[trajectory_eval_bridge,correctness_eval_bridge],
            experiment_prefix="agentevals-eval-test"
        )
      print("Expirment is done")

In [ ]:
await evaluate_dataset(dataset_name=dataset_name, tumor_type=tumor_type)

**Agent Correctness Evaluation**